In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import utils
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from dataset import load_uci_dataset, encode_labels, get_holdout_dataloaders, get_preprocessor
from models import MLP
from losses import PolyLoss, CombinedCLoss
from engine import train_model, evaluate_model
from plot_utils import plot_training_history
from gdv_utils import calculate_gdv
import random

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=== 1. SETUP AMBIENTE E DOWNLOAD DATI ===")

config_init = utils.carica_configurazione("config.yaml")
dataset_id = config_init['dataset']['uci_id']
device = torch.device(config_init['training']['device'] if torch.cuda.is_available() else "cpu")
print(f"Dispositivo in uso: {device}")

# Caricamento Dati
target_col = config_init['dataset'].get('target_column', None)
X_raw, y_raw = load_uci_dataset(dataset_id, target_column=target_col)

# Cattura il numero di feature PRIMA di qualsiasi pulizia (per il report finale)
in_dim_originale = X_raw.shape[1]

# =================================================================
# --- 1. SANIFICAZIONE E INFERENZA DEI TIPI AVANZATA (BISTURI) ---
# Sostituiamo stringhe vuote o segnaposto con veri NaN matematici
X_raw = X_raw.replace(['?', 'NA', 'null', 'None', '', ' '], np.nan)

for col in X_raw.columns:
    # Se la colonna NON è già matematicamente un numero (quindi è testo, object, string, ecc.)
    if not pd.api.types.is_numeric_dtype(X_raw[col]):

        # 1. Contiamo quanti dati REALI (non nulli) ci sono in questa colonna
        valori_presenti = X_raw[col].notna().sum()

        if valori_presenti > 0:
            # 2. Proviamo a convertire in numero, cambiando la virgola in punto SOLO per questo test
            tentativo = pd.to_numeric(X_raw[col].astype(str).str.replace(',', '.'), errors='coerce')
            numeri_validi = tentativo.notna().sum()

            # 3. Se il 90% dei dati *presenti* si è rivelato essere un numero...
            if (numeri_validi / valori_presenti) > 0.90:
                # ...allora la colonna è matematicamente numerica! La salviamo.
                X_raw[col] = tentativo
# =================================================================

# --- 2. BLACKLIST DATA LEAKAGE ---
# NOTA: La colonna 'result' è presente nel dataset Steel Plates Faults (UCI ID 198)
# come colonna di output binario già riassuntiva delle classi di difetto. Includerla
# tra le feature causerebbe data leakage diretto (la rete impara a memoria quella colonna).
# 'errors=ignore' garantisce che questa riga sia inoffensiva su tutti gli altri dataset.
BLACKLIST_LEAKAGE = ['result']
X_raw = X_raw.drop(columns=BLACKLIST_LEAKAGE, errors='ignore')

# --- 3. RIMOZIONE COLONNE ID E ALTA CARDINALITÀ ---
colonne_rimosse = []
for col in list(X_raw.columns):

    # Regola A: ID espliciti (tutti valori unici)
    if X_raw[col].nunique() == len(X_raw):

        # 1. Se è Testo (es. "Nome_Paziente", "Hash"), lo droppiamo subito
        if not pd.api.types.is_numeric_dtype(X_raw[col]):
            colonne_rimosse.append(col)
            X_raw = X_raw.drop(columns=[col])

        # 2. Se è un Intero (int), potrebbe essere una misura fisica (es. Pixel) o un ID (1,2,3...)
        elif pd.api.types.is_integer_dtype(X_raw[col]):
            # Gli ID veri sono quasi sempre sequenziali. Le misure fisiche no.
            if X_raw[col].is_monotonic_increasing or X_raw[col].is_monotonic_decreasing:
                colonne_rimosse.append(col)
                X_raw = X_raw.drop(columns=[col])

        # (I decimali continui - float - vengono ignorati e salvati automaticamente)

    # Regola B: Testo con alta cardinalità (>50% valori unici)
    elif not pd.api.types.is_numeric_dtype(X_raw[col]):
        if X_raw[col].nunique() > (len(X_raw) * 0.5):
            colonne_rimosse.append(col)
            X_raw = X_raw.drop(columns=[col])

if colonne_rimosse:
    print(f"[*] Attenzione: rimosse le seguenti colonne ID o ad alta cardinalità: {colonne_rimosse}")
# =================================================================

# Numero di feature DOPO la pulizia (prima del one-hot encoding)
in_dim_post_pulizia = X_raw.shape[1]

y_encoded, n_classes = encode_labels(y_raw)

print("Pulizia, codifica e calcolo GDV geometrico iniziale in corso...")

# NOTA sul GDV_INIZIALE: il preprocessor qui viene fittato sull'intero dataset
# (inclusi i dati che diventeranno val/test) per misurare la separabilità geometrica
# dei dati grezzi come grandezza di riferimento pre-training.
# Questo è intenzionale: GDV_INIZIALE misura lo spazio di INPUT, non lo spazio latente.
# I val_gdv calcolati durante il training misurano invece lo spazio LATENTE della rete,
# addestrata su dati scalati con un preprocessor fittato solo sul train set.
# I due indici non sono quindi confrontabili sulla stessa scala di normalizzazione,
# ma misurano concetti distinti: separabilità dei dati grezzi vs separabilità appresa.
preprocessor = get_preprocessor(X_raw)
X_processed = preprocessor.fit_transform(X_raw)

# --- SICUREZZA PYTORCH: Conversione da Matrice Sparsa a Densa ---
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

# --- Calcola la dimensione reale dopo il One-Hot Encoding ---
in_dim_processed = X_processed.shape[1]

# Conversione in tensori
X_tensor = torch.tensor(X_processed, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

GDV_INIZIALE = calculate_gdv(X_tensor, y_tensor)

print(f"\n--- Dimensioni Rilevate dal Dataset (UCI {dataset_id}) ---")
print(f"Features Originarie (raw):              {in_dim_originale}")
print(f"Features Post-Pulizia (pre-encoding):   {in_dim_post_pulizia}")
print(f"Features Post-Processing (Input Rete):  {in_dim_processed}")
print(f"Classi (Output):                        {n_classes}")
print(f"GDV Dati Grezzi (Spazio Input):         {GDV_INIZIALE:.4f}")
print("===================================================")

In [ ]:
print("=== 2. RICARICAMENTO CONFIGURAZIONE E DATALOADER ===")

config = utils.carica_configurazione("config.yaml")

# Imposta il seed per la riproducibilità degli split
random_state = config['dataset'].get('random_state', 42)
torch.manual_seed(random_state)

# --- CREAZIONE VELOCE DATALOADER ---
# X_raw e y_encoded sono prodotti dalla Cella 1 e già puliti/codificati
batch_size = config['dataset']['batch_size']
train_loader, val_loader, test_loader = get_holdout_dataloaders(
    X_raw, y_encoded, batch_size=batch_size, random_state=random_state
)

# --- IL VERO SALVAVITA: Leggiamo la dimensione dal Dataloader! ---
# Più robusto di X_processed.shape[1] perché legge la dimensione reale
# dei tensori che entreranno nella rete, dopo tutto il preprocessing.
batch_x, _ = next(iter(train_loader))
in_dim_processed = batch_x.shape[1]
print(f"\n[*] Feature REALI in uscita dal preprocessing (Train Set): {in_dim_processed}")

# --- SETUP ARCHITETTURA DINAMICA ---
config_layers = config['model'].get('hidden_layers', "auto")

if config_layers == "auto":
    num_samples = len(X_raw)
    hidden_layers = utils.get_auto_hidden_layers(in_dim_processed, n_classes, num_samples)
    print(f"Architettura Auto-Selezionata: {in_dim_processed} -> {hidden_layers} -> {n_classes} (su {num_samples} campioni)")
    config['model']['hidden_layers'] = hidden_layers
else:
    print(f"Architettura Forzata da YAML: {in_dim_processed} -> {config_layers} -> {n_classes}")

print(f"\nDataloader creati! (Batch size: {batch_size})")

# --- PESI DI CLASSE BILANCIATI (Hold-Out) ---
from sklearn.utils.class_weight import compute_class_weight
_y_train_ho = np.concatenate([lbl.numpy() for _, lbl in train_loader])
_cw = compute_class_weight('balanced', classes=np.arange(n_classes), y=_y_train_ho)
weights_holdout = torch.tensor(_cw, dtype=torch.float32).to(device)
print(f'[*] Pesi Hold-Out calcolati: {_cw.round(3)}')
print("Pronto per l'addestramento.")



In [ ]:
# === ESPERIMENTO 1: BASELINE (CROSS-ENTROPY) ===
print(f"\nAvvio Addestramento: Cross-Entropy su Dataset UCI ID {dataset_id} (Hold-Out)")

torch.manual_seed(config['dataset'].get('random_state', 42))

model_ce = MLP(
    input_size=in_dim_processed, 
    num_classes=n_classes,
    hidden_layers=config['model']['hidden_layers'],
    dropout_rate=config['model']['dropout_rate']
).to(device)

criterion_ce = nn.CrossEntropyLoss(weight=weights_holdout)
optimizer_ce = optim.Adam(model_ce.parameters(), lr=config['training']['learning_rate'])

gdv_layer = model_ce.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

history_ce, best_epoch_ce = train_model(
    model=model_ce, criterion=criterion_ce, optimizer=optimizer_ce, 
    train_loader=train_loader, val_loader=val_loader, device=device,
    num_epochs=config['training']['epochs'], layer_for_gdv=gdv_layer
)

test_acc_ce = evaluate_model(model_ce, test_loader, device)
history_ce['test_acc'] = test_acc_ce
print(f"Accuratezza Test Finale: {test_acc_ce:.2f}%")

# MLOps: Salvataggio log e Grafici
utils.salva_storico_json(     
    history=history_ce,     
    nome_esperimento=f"HoldOut_CE_UCI_{dataset_id}",
    dataset_id=dataset_id
)

In [ ]:
# === ESPERIMENTO 2: POLYLOSS ===
print(f"\nAvvio Addestramento: PolyLoss su Dataset UCI ID {dataset_id} (Hold-Out)")

eps = config.get('loss_params', {}).get('polyloss', {}).get('epsilon', 2.0)
nome_poly_plot = f"PolyLoss (e={eps}) - UCI {dataset_id}"

torch.manual_seed(config['dataset'].get('random_state', 42))

# Inizializzazione modello
model_poly = MLP(
    input_size=in_dim_processed, 
    num_classes=n_classes,
    hidden_layers=config['model']['hidden_layers'],
    dropout_rate=config['model']['dropout_rate']
).to(device)

criterion_poly = PolyLoss(epsilon=eps, weight=weights_holdout)
optimizer_poly = optim.Adam(model_poly.parameters(), lr=config['training']['learning_rate'])

gdv_layer = model_poly.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

history_poly, best_epoch_poly = train_model(
    model=model_poly, criterion=criterion_poly, optimizer=optimizer_poly, 
    train_loader=train_loader, val_loader=val_loader, device=device,
    num_epochs=config['training']['epochs'], layer_for_gdv=gdv_layer
)

test_acc_poly = evaluate_model(model_poly, test_loader, device)
history_poly['test_acc'] = test_acc_poly
print(f"Accuratezza Test Finale: {test_acc_poly:.2f}%")

# MLOps: Salvataggio log e Grafici
utils.salva_storico_json(     
    history=history_poly,     
    nome_esperimento=f"HoldOut_PolyLoss_UCI_{dataset_id}",
    dataset_id=dataset_id 
)

In [ ]:
print(f"\nAvvio Addestramento: Combined C-Loss su Dataset UCI ID {dataset_id} (Hold-Out)")

sig = config.get('loss_params', {}).get('closs', {}).get('sigma', 0.5)

nome_closs_plot = f"Combined C-Loss (s={sig}) - UCI {dataset_id}"

torch.manual_seed(config['dataset'].get('random_state', 42))

model_closs = MLP(
    input_size=in_dim_processed,
    num_classes=n_classes,
    hidden_layers=config['model']['hidden_layers'],
    dropout_rate=config['model']['dropout_rate']
).to(device)

criterion_closs = CombinedCLoss(sigma=sig, weight=weights_holdout)
optimizer_closs = optim.Adam(model_closs.parameters(), lr=config['training']['learning_rate'])

gdv_layer = model_closs.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

history_closs, best_epoch_closs = train_model(
    model=model_closs, criterion=criterion_closs, optimizer=optimizer_closs,
    train_loader=train_loader, val_loader=val_loader, device=device,
    num_epochs=config['training']['epochs'], layer_for_gdv=gdv_layer
)

test_acc_closs = evaluate_model(model_closs, test_loader, device)
history_closs['test_acc'] = test_acc_closs
print(f"Accuratezza Test Finale: {test_acc_closs:.2f}%")

utils.salva_storico_json(
    history=history_closs,
    nome_esperimento=f"HoldOut_CombinedCLoss_UCI_{dataset_id}",
    dataset_id=dataset_id
)

In [ ]:
print(f"=== Generazione Grafici Comparativi per Dataset {dataset_id} ===")

def get_global_limits(histories, initial_gdv=None):
    all_losses = []
    all_accs   = []
    all_gdvs   = []

    if initial_gdv is not None:
        all_gdvs.append(initial_gdv)

    for h in histories:
        all_losses.extend(h['train_loss'])
        all_losses.extend(h['val_loss'])
        all_accs.extend(h['val_acc'])
        if 'test_acc' in h:  all_accs.append(h['test_acc'])
        if 'val_gdv' in h:   all_gdvs.extend(h['val_gdv'])

    def get_lim(data, margin=0.05):
        low, high = min(data), max(data)
        diff = high - low
        # FIX: se diff==0 (metrica costante su tutte le epoche), aggiungiamo
        # un margine assoluto minimo per evitare un asse a range nullo in matplotlib.
        if diff == 0:
            return (low - 0.1, high + 0.1)
        return (low - diff * margin, high + diff * margin)

    return {
        'loss': get_lim(all_losses),
        'acc':  get_lim(all_accs),
        'gdv':  get_lim(all_gdvs)
    }

global_limits = get_global_limits([history_ce, history_poly, history_closs], initial_gdv=GDV_INIZIALE)

esperimenti = [
    (history_ce,    f"Cross-Entropy (UCI {dataset_id})"),
    (history_poly,  nome_poly_plot),
    (history_closs, nome_closs_plot)
]

for hist, name in esperimenti:
    plot_training_history(
        hist,
        model_name=name,
        gdv_iniziale=GDV_INIZIALE,
        dataset_id=dataset_id,
        y_limits=global_limits
    )

In [ ]:
import random
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from dataset import create_dataloader, get_preprocessor
from models import MLP

def imposta_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def run_kfold_experiment(loss_factory, model_name, X, y, n_classes, config):
    k_splits     = config.get('kfold', {}).get('k_splits', 10)
    epochs_kfold = config.get('kfold', {}).get('epochs', 20)
    batch_size   = config['dataset']['batch_size']
    base_seed    = config['dataset'].get('random_state', 42)

    print(f"\n=== AVVIO {k_splits}-FOLD CV: {model_name} ===")
    print(f"Configurazione: {epochs_kfold} epoche per fold")

    imposta_seed(base_seed)
    skf = StratifiedKFold(n_splits=k_splits, shuffle=True, random_state=base_seed)

    fold_accs = []
    fold_gdvs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{k_splits} ---")

        seed_corrente = base_seed + fold
        imposta_seed(seed_corrente)

        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]

        # --- 1. CALCOLO PESI BILANCIATI con protezione NaN ---
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=np.arange(n_classes),
            y=y_tr
        )
        # FIX: se una classe non appare in y_tr (fold sbilanciato su dataset piccoli),
        # compute_class_weight produce nan per quella classe.
        # nan nel tensore dei pesi → loss nan → gradienti nan → il fold non impara nulla.
        # Sostituiamo nan con 1.0: la classe assente non viene né premiata né penalizzata.
        nan_mask = np.isnan(class_weights)
        if nan_mask.any():
            classi_mancanti = np.where(nan_mask)[0].tolist()
            print(f"  [!] Fold {fold+1}: classi {classi_mancanti} assenti nel train. "
                  f"Peso impostato a 1.0 per evitare NaN nella loss.")
            class_weights = np.where(nan_mask, 1.0, class_weights)

        weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

        # --- 2. LOSS FACTORY ---
        loss_fn = loss_factory(weights_tensor)

        # --- 3. PREPROCESSING rigoroso (fit solo su train) ---
        preprocessor = get_preprocessor(X_tr)
        X_tr_proc = preprocessor.fit_transform(X_tr)
        X_va_proc = preprocessor.transform(X_va)

        fold_in_dim = X_tr_proc.shape[1]

        tr_loader = create_dataloader(X_tr_proc, y_tr, batch_size=batch_size, shuffle=True)
        va_loader = create_dataloader(X_va_proc, y_va, batch_size=batch_size, shuffle=False)

        model = MLP(
            input_size=fold_in_dim,
            num_classes=n_classes,
            hidden_layers=config['model']['hidden_layers'],
            dropout_rate=config['model']['dropout_rate']
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
        gdv_layer = model.get_layer_for_gdv(index=config['model']['gdv_layer_index'])

        history, best_ep = train_model(
            model=model, criterion=loss_fn, optimizer=optimizer,
            train_loader=tr_loader, val_loader=va_loader, device=device,
            num_epochs=epochs_kfold,
            layer_for_gdv=gdv_layer
        )

        acc = evaluate_model(model, va_loader, device)
        fold_accs.append(acc)

        best_gdv = history['val_gdv'][best_ep - 1] if history['val_gdv'] else 0.0
        fold_gdvs.append(best_gdv)

    return fold_accs, fold_gdvs


k_val = config.get('kfold', {}).get('k_splits', 10)
print(f"\n=== ESECUZIONE {k_val}-FOLD CV SU UCI ID {dataset_id} ===")

eps = config.get('loss_params', {}).get('polyloss', {}).get('epsilon', 2.0)
sig = config.get('loss_params', {}).get('closs', {}).get('sigma', 0.5)

nome_poly  = f"PolyLoss (e={eps})"
nome_closs = f"C-Loss (s={sig})"

acc_ce,    gdv_ce    = run_kfold_experiment(
    lambda w: nn.CrossEntropyLoss(weight=w),
    "Cross-Entropy", X_raw, y_encoded, n_classes, config
)
acc_poly,  gdv_poly  = run_kfold_experiment(
    lambda w: PolyLoss(epsilon=eps, weight=w),
    nome_poly, X_raw, y_encoded, n_classes, config
)
acc_closs, gdv_closs = run_kfold_experiment(
    lambda w: CombinedCLoss(sigma=sig, weight=w),
    nome_closs, X_raw, y_encoded, n_classes, config
)

kfold_results = {
    "dataset_id": dataset_id,
    "folds": k_val,
    "Cross-Entropy": {"acc": acc_ce,   "gdv": gdv_ce},
    nome_poly:       {"acc": acc_poly,  "gdv": gdv_poly},
    nome_closs:      {"acc": acc_closs, "gdv": gdv_closs}
}

utils.salva_storico_json(
    history=kfold_results,
    nome_esperimento="KFold_Results",
    dataset_id=dataset_id
)

print("\n" + "="*50)
print("--- RIEPILOGO FINALE K-FOLD (Accuratezza Media) ---")
print(f"1. Cross-Entropy: {np.mean(acc_ce):.2f}% (± {np.std(acc_ce):.2f}%)")
print(f"2. {nome_poly}:   {np.mean(acc_poly):.2f}% (± {np.std(acc_poly):.2f}%)")
print(f"3. {nome_closs}:  {np.mean(acc_closs):.2f}% (± {np.std(acc_closs):.2f}%)")

In [ ]:
import os
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(5.5, 4.0))

modelli = ['Cross-Entropy', 'PolyLoss', 'Combined C-Loss']
medie_acc = [np.mean(acc_ce), np.mean(acc_poly), np.mean(acc_closs)]
std_acc = [np.std(acc_ce), np.std(acc_poly), np.std(acc_closs)]

colori = ['#004488', '#BB5500', '#117733']

barre = ax1.bar(modelli, medie_acc, yerr=std_acc, capsize=5, 
                color=colori, alpha=0.85, edgecolor='black', linewidth=1.0,
                error_kw={'elinewidth': 1.0, 'capthick': 1.0})

# ==========================================
# FIX: ASSI FISSI PER CONFRONTO GLOBALE
# ==========================================
y_min_fisso = 40.0  # Baseline fissa per avere proporzioni reali
y_max_fisso = 100.0

# Gestione eccezioni: se un dataset dovesse andare peggio del 40%, abbassiamo la visuale solo per lui
if min(medie_acc) < y_min_fisso:
    y_min_fisso = max(0.0, min(medie_acc) - 10.0)

ax1.set_ylim(y_min_fisso, y_max_fisso)

ax1.set_ylabel('Accuratezza Media (%)')
ax1.set_title(f'Confronto Prestazioni 10-Fold Cross Validation\n(Dataset UCI ID: {dataset_id})')

# Ricalcolo della centratura del testo rispetto al nuovo asse
for i, (barra, media) in enumerate(zip(barre, medie_acc)):
    altezza = barra.get_height()
    base_error_bar = altezza - std_acc[i]
    
    # Il testo si posiziona esattamente a metà tra il fondo del grafico e la sbarra d'errore
    y_text = y_min_fisso + (base_error_bar - y_min_fisso) / 2
    
    # Sicurezza per evitare che il testo "buchi" il fondo in casi estremi
    if y_text < y_min_fisso + 2:
        y_text = y_min_fisso + 5
    
    ax1.text(barra.get_x() + barra.get_width()/2., y_text, 
             f'{media:.2f}%', ha='center', va='center', 
             color='white', fontweight='bold', fontsize=10)

plt.grid(axis='y', linestyle='--')
plt.tight_layout()

# ==========================================
# SALVATAGGIO MLOPS
# ==========================================
base_dir = "results"
save_dir = os.path.join(base_dir, str(dataset_id), "plots")
os.makedirs(save_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename_pdf = os.path.join(save_dir, f"{timestamp}_KFold_Acc.pdf")

plt.savefig(filename_pdf, format='pdf', bbox_inches='tight')
print(f"[*] Grafico K-Fold salvato in: {filename_pdf}")

plt.show()